# SALM Fine-tuning 01 · Teach it a new language: Swahili ASR

> **Task** speech → Swahili transcript · **Data** FLEURS `sw_ke` (13.5 h) · **Metric** WER ↓
> **Result** 81.16% → **18.01%** WER (−77.8%) · **Time** ~1.5 h on one GB300

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: a polyglot with a blind spot

The model transcribes many languages, but not all of them. To find out which ones it knows, decode a language's dev
set with a plain prompt ("Transcribe the audio.") and look at the WER. For Swahili it is **99.2%** on FLEURS dev,
nearly one error per reference word. The model doesn't go silent, it **drifts**: Swahili slides into Arabic script
mid-sentence:

```
REF   Amazon pia ni mto mpana zaidi Duniani, wakati mwingine maili sita kwa upana.
BASE  Amazon بيئة محيطة بانها زايدى دنيانى، وكانت يمكن ما يلي سى تا كوبا.
```

Swahili uses the Latin script and its spelling follows pronunciation, and FLEURS has 13.5 h of training audio. The
goal is a usable Swahili transcriber that keeps its English.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data

`prepare_benchmarks.py` downloads FLEURS Swahili and writes 16 kHz FLAC plus NeMo manifests: one JSON object per line
with `audio_filepath`, `duration`, `text` and `context` (the prompt). Manifests must end in `.json`: `.jsonl` means
*Lhotse* manifest to NeMo's loader.

In [ ]:
sh(f"{PYTHON} {FT}/prepare_benchmarks.py fleurs --langs sw_ke", log="prep_fleurs.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")   # the general-English guard
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(M / "fleurs" / f"sw_ke_{s}.json")]
    print(f"{s:5s} {len(rows):5d} utterances, {sum(r['duration'] for r in rows) / 3600:.2f} h")
print(rows[0])

## 2. A fair baseline: prompting is worth 20 WER points

Before training anything, give the base model its best shot: three prompts, scored on dev **and** test. The baseline
is the test score of the prompt chosen on **dev**, the same way the fine-tuned checkpoint is chosen. The
**test-oracle** (the best prompt on test) is printed too, as an upper bound for prompting alone.

In [ ]:
TASK = "--task asr"
PROMPTS = {
    "p1": "Transcribe the audio.",
    "p2": "Transcribe the Swahili audio.",
    "p3": "Provide a verbatim transcript of the audio in Swahili (Kiswahili).",
}
base = {}
for p, text in PROMPTS.items():
    for s in ("dev", "test"):
        r = evaluate(CKPT, M / "fleurs" / f"sw_ke_{s}.json", f"base_{p}_{s}", TASK, f"--prompt {q(text)}")
        base[p, s] = 100 * r["score"]
    print(f"{p}  dev WER {base[p, 'dev']:6.2f}%   test WER {base[p, 'test']:6.2f}%   {text}")
BEST_P = min(PROMPTS, key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = min(base[p, "test"] for p in PROMPTS)  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}% WER")
samples("base_p3_dev", n=3)

Naming the language removes ~20 points of error (dev WER 99.2% → 76.3%). The two Swahili prompts tie on dev (76.31%
vs 76.23%). The verbatim one (p3) wins by a hair, so it is the prompt chosen on dev: the baseline is **81.16%** on
test, and p3 is the training prompt. On test the shorter prompt does better: the test-oracle is 77.65%.

What's left is mostly **spelling**: the model writes what it hears, e.g. `luga` for `lugha`, `kutoka na nakukaliwa`
for `kutokana na kukaliwa`. That's exactly what fine-tuning is good at.

The next cell writes the prompt into every manifest row's `context` field. The trainer (`lhotse_as_conversation`) and
the evaluator both read that field, so the training and evaluation prompts can't drift apart.

In [ ]:
d = M / "fleurs"
pairs = " ".join(f"{d}/sw_ke_{s}.json {d}/sw_ke_{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(PROMPTS['p3'])} {pairs}")

## 3. The config: derived from the checkpoint

Never start from an example YAML. It describes a *different* model, and a mismatched encoder config loads silently
with random weights. `make_ft_config.py` copies the checkpoint's own `config.json` and replaces only what a fine-tune
owns:

* **freeze policy**: `--train-encoder --train-proj`. Swahili sounds are new to the acoustic front end, so the speech
  encoder learns; the LLM stays frozen behind LoRA adapters (r=32, α=64).
* **`--prompt manifest`**: read each row's `context`.
* **batching**: 13.5 h is small, so size batches for optimizer *steps*, not throughput. `epoch_batches.py` counts
  the real batches per epoch. `limit_train_batches` must stay *below* that count, or validation silently never runs.

In [ ]:
sh(f"{PYTHON} {FT}/epoch_batches.py {d}/sw_ke_train_ft.json 24000 --tokenizer {CKPT}")
CFG = WORK / "configs" / "swahili.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'swahili'} "
   f"--train-manifest {d}/sw_ke_train_ft.json --val-manifest {d}/sw_ke_dev_ft.json --val-name sw_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 60 --max-steps 600 "
   f"--batch-tokens 24000 --limit-train-batches 30 --val-every-n-epochs 2 --limit-val-batches 30 --save-top-k -1")

## 4. Preflight, smoke test, train

Two checks catch this stack's worst *silent* failures before you spend hours training:

* `check_checkpoint_coverage.py`: every checkpoint tensor must find a home in the configured model (here 1,269 of
  1,269) and the only new parameters must be LoRA adapters.
* `run_training.py --smoke`: 20 steps, then it asserts the restore log is clean and the loss is finite.

The full run trains with a watchdog that stops it if validation never happens. It ends at step 420
(`trainer.max_steps=420`) but keeps the 600-step LR schedule, and keeps only the checkpoints evaluated below:
~2.3 s/step, ~20 min.

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 120 240 420 trainer.max_steps=420")

Expected `val_loss` at steps 120 / 240 / 420: about 0.49 / 0.68 / 0.77. `val_loss` bottoms out at step 120 and climbs
from there, while dev WER keeps improving (below). Rising `val_loss` here means over-confidence on the few tokens the
model still gets wrong, not worse transcripts. **Always select by decoding.**

## 5. Select on dev: export, merge, decode, average

`export_checkpoint.py` converts a training checkpoint into a HuggingFace directory and **merges the LoRA adapters
into the weights**, so no serving stack can silently ignore them. It also asserts that `lora_B` ≠ 0, which proves
the trained weights made it in.

In [ ]:
dev_wer = {}
for step in (120, 240, 420):
    out = EXPORT / f"swahili_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'swahili'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_swahili_{step}.log")
    dev_wer[step] = 100 * evaluate(out, d / "sw_ke_dev_ft.json", f"swahili_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev WER {dev_wer[step]:.2f}%")

In [ ]:
AVG = EXPORT / "swahili_avg240_420"
sh(f"{PYTHON} {FT}/average_checkpoints.py --src {EXPORT / 'swahili_s240'} {EXPORT / 'swahili_s420'} --dst {AVG} --reuse")
dev_wer["avg(240,420)"] = 100 * evaluate(AVG, d / "sw_ke_dev_ft.json", "swahili_avg_dev", TASK)["score"]
for k, v in dev_wer.items():
    print(f"{str(k):>14}: dev WER {v:.2f}%")

Expected dev WER: step 120 → 21.5%, step 240 → 20.1%, step 420 → 18.1%, **average(240, 420) → 18.8%**. Averaging two
checkpoints of one run (they sit in the same loss basin) is usually a free gain in this series. Here the average lands
between its parents, 0.7 points behind step 420 on dev and well inside the run-to-run spread. It is the selected
model.

## 6. Test, once, plus the English guard

Training is not bit-for-bit reproducible; expect the fine-tuned test WER to vary by about ±2 points run to run (the
check below allows ±2.5).

In [ ]:
ft = evaluate(AVG, d / "sw_ke_test_ft.json", "swahili_avg_TEST", TASK)
guard = evaluate(AVG, M / "librispeech" / "test_clean.json", "swahili_avg_librispeech", "--task asr --normalizer english")
base_guard = evaluate(CKPT, M / "librispeech" / "test_clean.json", "base_librispeech", "--task asr --normalizer english")
FT_WER = 100 * ft["score"]
print(f"test WER: base {BASELINE:.2f}%  →  fine-tuned {FT_WER:.2f}%   ({(BASELINE - FT_WER) / BASELINE:+.1%} relative reduction)")
print(f"LibriSpeech WER: base {100 * base_guard['score']:.2f}%  →  fine-tuned {100 * guard['score']:.2f}%")
samples("swahili_avg_TEST", n=3)
result("01 baseline test WER", BASELINE, 81.16, 0.5)
result("01 fine-tuned test WER", FT_WER, 18.01, 2.5)

Expected results:

| | test WER | LibriSpeech WER |
|---|---:|---:|
| base, prompt chosen on dev (p3) | 81.16% | 1.54% |
| **fine-tuned** | **18.01%** | 1.70% |

**−77.8% relative** (−76.8% against the test-oracle prompt, 77.65%). English barely changes, and the spelling is
Swahili:

```
REF   Kuna fukwe nyingi, kutokana na kukaliwa magamaga kwa kwa bandari mbili na Auckland.
BASE  Kuna fuque nyingi kutoka na nakukaliwa magaunga kwa maandaa rindi au na Auckland
FT    Kuna fukwe nyingi kutokana na kukaliwa magamaga kwa mabandari mbili au na Auckland
```

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=AVG, name="swahili")

## Takeaways

1. **Probe coverage first.** "New language" means different things for a language the model half-knows and one it
   has never heard.
2. **Name the language in the prompt.** It's free and worth 20 points. Put the same prompt in training.
3. **The error was orthography, not hearing.** A strong multilingual encoder was one step away.
4. **`val_loss` ≠ WER.** Decode your checkpoints, and average a couple.